# Dataset 2 — Centrality threshold sensitivity (XGBoost tuned)

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.metrics import mean_absolute_error, mean_squared_error

def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        if (p / 'src').exists() and (p / 'requirements.txt').exists():
            return p
    raise FileNotFoundError('Project root not found.')

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))
from src.models.ml_train_and_store import load_model, random_split, CLASSICAL_FEATURE_CANDIDATES

DATASET = 'dataset_2'
APPROACH = 'centrality'
TARGET_COL = 'log_systemic_risk_label'
P_LIST = ['p0', 'p5', 'p10', 'p15', 'p20', 'p25', 'p30', 'p35', 'p40']

TARGETS_DIR = PROJECT_ROOT / 'src' / 'datasets' / 'dataset_2' / 'targets'
SRC = PROJECT_ROOT / 'src' / 'data' / 'classical_features' / 'dataset_2' / 'classical_features_dataset2.parquet'
MODEL_PATH  = PROJECT_ROOT / 'src' / 'models' / 'dataset_2/04_a/XGBoost_(tuned).joblib'
OUT_DIR     = PROJECT_ROOT / 'src' / 'data' / 'predictions' / 'dataset_2/centrality_threshold'
OUT_DIR.mkdir(parents=True, exist_ok=True)

model = load_model(MODEL_PATH)
print('model:', MODEL_PATH.name, '| approach:', APPROACH)

model: XGBoost_(tuned).joblib | approach: centrality


## Apply the selected model across thresholds

In [2]:
def load_p(p):
    tname = 'target.csv' if p == 'p0' else f'target_{p}.csv'
    target = pd.read_csv(TARGETS_DIR / tname)
    df = pd.read_parquet(SRC).merge(target, on='bank_id', how='inner')
    fcols = [c for c in CLASSICAL_FEATURE_CANDIDATES if c in df.columns]
    return df.dropna(subset=[TARGET_COL]).reset_index(drop=True), fcols

In [3]:
rows, preds = [], []
for p in P_LIST:
    df, fcols = load_p(p)
    tr, va, te = random_split(df, TARGET_COL)            # 70/15/15 stratified
    m = clone(model).fit(tr[fcols], tr[TARGET_COL])      # same selected model, refit at this threshold
    row = {'p': p}
    for split_name, sdf in [('train', tr), ('validation', va), ('test', te)]:
        yp = m.predict(sdf[fcols])
        row[f'{split_name}_rmse'] = mean_squared_error(sdf[TARGET_COL], yp) ** 0.5
        row[f'{split_name}_mae']  = mean_absolute_error(sdf[TARGET_COL], yp)
        pf = sdf[['bank_id', TARGET_COL]].copy()
        pf['prediction'] = yp; pf['split'] = split_name; pf['p'] = p
        pf['dataset'] = DATASET; pf['approach'] = APPROACH
        preds.append(pf)
    row['val/train_rmse'] = round(row['validation_rmse'] / row['train_rmse'], 2)
    row['val/train_mae']  = round(row['validation_mae'] / row['train_mae'], 2)
    rows.append(row)

metrics = pd.DataFrame(rows)
predictions = pd.concat(preds, ignore_index=True)
metrics.to_csv(OUT_DIR / 'metrics.csv', index=False)
predictions.to_csv(OUT_DIR / 'predictions.csv', index=False)
print('saved ->', OUT_DIR)
display(metrics.round(3))

saved -> /Users/rubenmarques/Documents/Repositórios/Thesis/src/data/predictions/dataset_2/centrality_threshold


,p,train_rmse,train_mae,validation_rmse,validation_mae,test_rmse,test_mae,val/train_rmse,val/train_mae
0,p0,0.149,0.049,0.157,0.058,0.164,0.054,1.05,1.19
1,p5,0.149,0.052,0.156,0.059,0.173,0.059,1.04,1.15
2,p10,0.154,0.050,0.191,0.070,0.180,0.056,1.25,1.40
3,p15,0.157,0.053,0.210,0.074,0.190,0.059,1.34,1.39
4,p20,0.166,0.062,0.212,0.077,0.202,0.066,1.28,1.25
5,p25,0.166,0.062,0.179,0.071,0.186,0.070,1.08,1.15
6,p30,0.170,0.060,0.289,0.101,0.201,0.072,1.70,1.67
7,p35,0.171,0.067,0.229,0.088,0.206,0.070,1.34,1.31
8,p40,0.180,0.075,0.269,0.094,0.223,0.065,1.49,1.25
